# Step 1 Real Profiling — UnifoLM-VLA-0 (G1_Clean_Table)

Runs **actual model profiling** for `unitreerobotics/UnifoLM-VLM-Base` on `G1_Clean_Table`.

Uses `torch.profiler` to trace per-operation CPU and GPU execution time during VLA action generation, then runs the full latency benchmark.

**V100 runtime optimizations** (Tesla V100 / Volta SM 7.0) are enabled by default in `UnifoLMVLAWrapper`:

| Optimization | Implementation |
|---|---|
| Native FP16 Tensor Cores | `torch.float16` load + `autocast(fp16)` |
| Kernel fusion | `torch.compile(..., mode="reduce-overhead")` |
| PCIe Gen 3 H2D | `pin_memory()` + `.to(cuda, non_blocking=True)` |
| No in-loop sync | `infer_gpu()` keeps actions on GPU; `infer()` syncs only at logging boundary |
| 100 Hz ESN path | `create_async_runtime()` + `GPUActionRegister.sample()` |

All artifacts are saved with a timestamped run tag, and each report includes `profiler_source` so you can distinguish PyTorch-profiler results from Nsight Systems results.

For hardware-level timeline capture with NVIDIA Nsight Systems, use `research/notebooks/step1_unifolm_nsight_systems_profiling.ipynb`.

## Before you run

| Requirement | Notes |
|-------------|-------|
| GPU | NVIDIA CUDA (same as OpenVLA notebook) |
| `transformers` | **>= 4.49** (Qwen2.5-VL). **Not** `requirements-gpu.txt` (OpenVLA pins 4.40) |
| `jinja2` | **>= 3.1.0** (required for `apply_chat_template`) |
| Disk | ~15+ GB for HF cache + weights |

Use a **separate venv** from OpenVLA:

```bash
cd research_summer_2026/research
python3 -m venv .venv-unifolm && source .venv-unifolm/bin/activate
pip install -U pip
pip install -r requirements-unifolm-gpu.txt
python -m ipykernel install --user --name unifolm_g1 --display-name "UnifoLM G1"
```

Select kernel **UnifoLM G1** in Jupyter before running.

**CLI equivalent:**
```bash
python3 -m src.step1_profile_unifolm_vla0 --n_trials 100 --no-mock-fallback
```

In [1]:
from pathlib import Path
import os
import sys
import shutil

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    RESEARCH_DIR = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "src" / "step1_profile_unifolm_vla0.py").is_file():
    RESEARCH_DIR = NOTEBOOK_DIR
else:
    RESEARCH_DIR = NOTEBOOK_DIR / "research_summer_2026" / "research"
    if not RESEARCH_DIR.is_dir():
        RESEARCH_DIR = NOTEBOOK_DIR / "research"

RESEARCH_DIR = RESEARCH_DIR.resolve()
assert (RESEARCH_DIR / "src").is_dir(), f"src package not found under: {RESEARCH_DIR}"

os.chdir(RESEARCH_DIR)
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

print(f"Research root : {RESEARCH_DIR}")
print(f"Results base  : {RESEARCH_DIR / 'results' / 'step1_profiling_unifolm_vla0'}")
print(f"Each run saves to: <results_base>/<profiler_source>_YYYYMMDD_HHMMSS/")
disk_gb = shutil.disk_usage(RESEARCH_DIR).free / (1024 ** 3)
print(f"Free disk     : {disk_gb:.1f} GB")

Research root : /home/aihimekpen/research_summer_2026/research
Results base  : /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0
Each run saves to: <results_base>/<profiler_source>_YYYYMMDD_HHMMSS/
Free disk     : 42.6 GB


In [2]:
import shutil
import torch

from src.step1_profile_unifolm_vla0 import (
    UNIFOLM_MIN_JINJA2,
    UNIFOLM_MIN_TRANSFORMERS,
    _unifolm_jinja2_ok,
    _unifolm_transformers_ok,
)

checks_ok = True

if not torch.cuda.is_available():
    checks_ok = False
    print("FAIL: CUDA not available.")
else:
    props = torch.cuda.get_device_properties(0)
    print(f"OK  : GPU {props.name} ({props.total_memory / (1024 ** 3):.1f} GB VRAM)")

try:
    import transformers

    ver = transformers.__version__
    if _unifolm_transformers_ok():
        print(f"OK  : transformers {ver} (>= {'.'.join(map(str, UNIFOLM_MIN_TRANSFORMERS))})")
    else:
        checks_ok = False
        print(
            f"FAIL: transformers {ver} is too old for UnifoLM (Qwen2.5-VL). "
            f"Need >= {'.'.join(map(str, UNIFOLM_MIN_TRANSFORMERS))}. "
            "pip install -r requirements-unifolm-gpu.txt in a fresh venv."
        )
except ImportError:
    checks_ok = False
    print("FAIL: transformers not installed")

disk_gb = shutil.disk_usage(RESEARCH_DIR).free / (1024 ** 3)
print(f"OK  : {disk_gb:.0f} GB free disk")
if disk_gb < 15:
    print("WARN: < 15 GB free — model download may fail")

assert checks_ok, "Fix failed checks (transformers >= 4.49, jinja2 >= 3.1.0) before continuing."

/home/aihimekpen/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


OK  : GPU Tesla V100-DGXS-32GB (31.7 GB VRAM)
OK  : transformers 4.49.0 (>= 4.49.0)
OK  : 43 GB free disk


In [3]:
from datetime import datetime

MOCK = False
N_TRIALS = 100
N_WARMUP = 2
N_PROFILE_STEPS = 3
MODEL_ID = "unitreerobotics/UnifoLM-VLM-Base"
INSTRUCTION = "Clean the table by moving all clutter items into the bin."

# Task focus for now: wipe/clean table. Update this when moving to other tasks.
TASK_LABEL = "G1_Clean_Table"
PROFILER_SOURCE = "pytorch_profiler"
RUN_TAG = f"{PROFILER_SOURCE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"MOCK={MOCK} | N_TRIALS={N_TRIALS}")
print(f"torch.profiler warmup={N_WARMUP} | profile_steps={N_PROFILE_STEPS}")
print(f"MODEL_ID={MODEL_ID}")
print(f"TASK_LABEL={TASK_LABEL}")
print(f"PROFILER_SOURCE={PROFILER_SOURCE}")
print(f"RUN_TAG={RUN_TAG}")

MOCK=False | N_TRIALS=100
torch.profiler warmup=2 | profile_steps=3
MODEL_ID=unitreerobotics/UnifoLM-VLM-Base
TASK_LABEL=G1_Clean_Table
PROFILER_SOURCE=pytorch_profiler
RUN_TAG=pytorch_profiler_20260617_123054


In [4]:
import importlib
import src.step1_profile_unifolm_vla0 as _step1_unifolm

importlib.reload(_step1_unifolm)

from src.step1_profile_unifolm_vla0 import (
    MockG1CleanTableEnv,
    UnifoLMVLAWrapper,
    RESULTS_BASE_DIR,
    make_run_results_dir,
    run_vla_torch_profiler,
    profile_unifolm_vla0,
    save_logs,
    plot_profiling_report,
    _normalize_unifolm_config,
)

assert callable(_normalize_unifolm_config), (
    "Stale module cache: restart kernel (Kernel -> Restart) and run all cells."
)

RUN_RESULTS_DIR = make_run_results_dir(PROFILER_SOURCE, RUN_TAG)

env = MockG1CleanTableEnv(image_size=(224, 224))
model = UnifoLMVLAWrapper(
    model_id=MODEL_ID,
    use_int4=False,
    action_dim=29,
    allow_mock_fallback=MOCK,
)

print(f"Model ready: {model.model_id} | mock={model.model is None}")
print(f"Run tag      : {RUN_TAG}")
print(f"Run folder   : {RUN_RESULTS_DIR}")

2026-06-17 12:30:54,650 [INFO] MockG1CleanTableEnv initialised | DOF=29 | img=(224, 224)
2026-06-17 12:30:54,798 [INFO] Loading model: unitreerobotics/UnifoLM-VLM-Base | INT4=False | FP16=True | compile=True
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.90it/s]
2026-06-17 12:31:01,261 [INFO] Loaded with Qwen2_5_VLForConditionalGeneration
2026-06-17 12:31:04,648 [INFO] torch.compile enabled (mode=reduce-overhead)
2026-06-17 12:31:04,649 [INFO] Model loaded successfully from HuggingFace.
2026-06-17 12:31:14,883 [INFO] Warmup complete (2 passes).


Model ready: unitreerobotics/UnifoLM-VLM-Base | mock=False
Run tag      : pytorch_profiler_20260617_123054
Run folder   : /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/pytorch_profiler_20260617_123054


In [ ]:
# torch.profiler: per-operation CPU/GPU trace for VLA action generation
profiler_out = run_vla_torch_profiler(
    model=model,
    env=env,
    instruction=INSTRUCTION,
    n_warmup=N_WARMUP,
    n_profile_steps=N_PROFILE_STEPS,
    profiler_source=PROFILER_SOURCE,
    run_tag=RUN_TAG,
    results_dir=RUN_RESULTS_DIR,
)

print("Top operations (GPU time when CUDA is available, else CPU):")
print(profiler_out["table"])
print("\nProfiler source:", profiler_out["report"]["profiler_source"])
print("Run folder     :", profiler_out["results_dir"])
print("Chrome trace   :", profiler_out["trace_path"])
print("Ops table      :", profiler_out["ops_txt_path"])
print("Ops JSON       :", profiler_out["ops_json_path"])

2026-06-17 12:31:14,897 [INFO] torch.profiler warmup | n_warmup=2 | n_profile_steps=3 | GPU=True
[W617 12:31:34.168711419 CPUAllocator.cpp:249] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event


In [ ]:
report, records = profile_unifolm_vla0(
    model=model,
    env=env,
    n_trials=N_TRIALS,
    instruction=INSTRUCTION,
    profiler_source=PROFILER_SOURCE,
    run_tag=RUN_TAG,
)

log_paths = save_logs(report, records)
fig_paths = plot_profiling_report(report, records)

print("\n" + "=" * 60)
print("  PROFILING COMPLETE — Week 1-2 Deliverable")
print("=" * 60)
print(f"  Model          : {report.model_id}")
print(f"  Task           : {report.task}")
print(f"  Mean latency   : {report.mean_latency_ms:.1f} +- {report.std_latency_ms:.1f} ms")
print(f"  Control rate   : {report.mean_hz:.2f} Hz  (target: 100 Hz)")
print(f"  Frequency gap  : {report.frequency_gap:.1f}x")
print(f"  Failure rate   : {report.failure_rate*100:.1f}%")
print(f"  Profiler source: {report.profiler_source}")
print(f"  Generated at   : {report.generated_at}")
print(f"  Run tag        : {report.run_tag}")
print(f"  Outputs        : {RUN_RESULTS_DIR.resolve()}")
print("=" * 60)

2026-06-17 11:11:43,778 [INFO] Starting profiling | task=G1_Clean_Table | n_trials=100 | GPU=True
Profiling UnifoLM-VLA-0: 100%|██████████| 100/100 [11:32<00:00,  6.92s/it]
2026-06-17 11:23:16,030 [INFO] Logs saved: /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/inference_log.json, /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/profiling_report.json, /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/profiling_summary.txt
2026-06-17 11:23:17,796 [INFO] Figure saved: /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/unifolm_vla0_profiling_report.pdf



  PROFILING COMPLETE — Week 1-2 Deliverable
  Model          : unitreerobotics/UnifoLM-VLM-Base
  Task           : G1_Clean_Table
  Mean latency   : 6920.2 +- 5.7 ms
  Control rate   : 0.14 Hz  (target: 100 Hz)
  Frequency gap  : 692.0x
  Failure rate   : 47.0%
  Outputs        : /home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0


In [ ]:
summary_path = log_paths["summary_path"]
report_path = log_paths["report_path"]
inference_log_path = log_paths["log_path"]
fig_pdf_path = fig_paths["pdf_path"]
fig_png_path = fig_paths["png_path"]
trace_path = profiler_out["trace_path"]
ops_txt_path = profiler_out["ops_txt_path"]
ops_json_path = profiler_out["ops_json_path"]

print("Profiler source:", report.profiler_source)
print("Run tag        :", report.run_tag)
print(inference_log_path)
print(summary_path)
print(report_path)
print(fig_pdf_path)
print(fig_png_path)
print(trace_path)
print(ops_txt_path)
print(ops_json_path)
print(f"Inference log exists: {inference_log_path.exists()}")
print(f"Summary exists      : {summary_path.exists()}")
print(f"Report exists       : {report_path.exists()}")
print(f"Figure PDF exists   : {fig_pdf_path.exists()}")
print(f"Figure PNG exists   : {fig_png_path.exists()}")
print(f"Trace exists        : {trace_path.exists()}")
print(f"Ops txt exists      : {ops_txt_path.exists()}")
print(f"Ops json exists     : {ops_json_path.exists()}")

/home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/profiling_summary.txt
/home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/profiling_report.json
/home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/unifolm_vla0_profiling_report.png
/home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/vla_action_chrome_trace.json
/home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/vla_action_profiler_ops.txt
/home/aihimekpen/research_summer_2026/research/results/step1_profiling_unifolm_vla0/vla_action_profiler_ops.json
Summary exists: True
Report exists : True
Figure exists : True
Trace exists  : True
Ops txt exists: True
Ops json exists: True


In [ ]:
# Optional: dual-stream async runtime for 100 Hz ESN integration
# VLA runs on a background CUDA stream; ESN samples GPUActionRegister on the main thread.

import time
import torch

runtime = model.create_async_runtime(instruction=INSTRUCTION, auto_start=True)

# Simulate robot loop: push observations, ESN ticks at 100 Hz without blocking on VLA
obs = env.reset()
for tick in range(20):
    runtime.submit_observation(obs, joint_state=env.joint_pos)
    vla_action_gpu = model.action_register.sample()  # non-blocking GPU read
    # esn_output = esn_control_tick(model.action_register, esn_step_fn=your_esn)
    time.sleep(0.01)  # 100 Hz
    obs, _, done, _ = env.step(vla_action_gpu.cpu().numpy())  # env.step needs CPU; ESN stays GPU-only
    if done:
        obs = env.reset()

runtime.stop()
print(f"Async VLA ticks: {runtime.ticks} | register publishes: {model.action_register.sequence}")